In [1]:
# test_edgepush.py
import numpy as np
from aad.core.seeds import grads
from aad.core.engine import hvp_for, edge_push_hessian
from aad.ops import mul, div, pow, norm_cdf, exp, log, sqrt

from aad.core.seeds import grads

def check_f(f, inputs, name):
    print(f"\n==== Test {name} ====")

    # gradients
    g = grads(f, inputs)
    print("grads:", g)

    # FoR
    n = len(inputs)
    H_for = []
    for i, k in enumerate(inputs.keys()):
        e = {kk: 0.0 for kk in inputs.keys()}
        e[k] = 1.0
        hv = hvp_for(f, inputs, e)
        H_for.append(hv)
    H_for = np.column_stack(H_for)

    # EP
    H_ep = edge_push_hessian(f, inputs, sparse=False)

    print("H (FoR rebuilt):\n", H_for)
    print("H (Edge-Pushing):\n", H_ep)
    print("diff:\n", H_ep - H_for)

def main():
    # f(x,z) = x * z
    f1 = lambda d: d["x"] * d["z"]
    check_f(f1, {"x": 2.0, "z": 3.0}, "mul")

    # f(x,z) = x / z
    f2 = lambda d: d["x"] / d["z"]
    check_f(f2, {"x": 2.0, "z": 3.0}, "div")

    # f(x,p) = x^p
    f3 = lambda d: d["x"] ** d["p"]
    check_f(f3, {"x": 2.0, "p": 3.0}, "pow")

    # f(x) = Φ(x) (norm_cdf)
    f4 = lambda d: norm_cdf(d["x"])
    check_f(f4, {"x": 1.0}, "norm_cdf")

    # f(x) = exp(x) + log(x)
    f5 = lambda d: exp(d["x"]) + log(d["x"])
    check_f(f5, {"x": 2.0}, "exp+log (mixed)")

if __name__ == "__main__":
    main()



==== Test mul ====
grads: {'x': np.float64(3.0), 'z': np.float64(2.0)}
H (FoR rebuilt):
 [[0. 1.]
 [1. 0.]]
H (Edge-Pushing):
 [[0. 1.]
 [1. 0.]]
diff:
 [[0. 0.]
 [0. 0.]]

==== Test div ====
grads: {'x': np.float64(0.3333333333333333), 'z': np.float64(-0.2222222222222222)}
H (FoR rebuilt):
 [[ 0.         -0.11111111]
 [-0.11111111  0.14814815]]
H (Edge-Pushing):
 [[ 0.         -0.11111111]
 [-0.11111111  0.14814815]]
diff:
 [[0. 0.]
 [0. 0.]]

==== Test pow ====
grads: {'x': np.float64(12.0), 'p': np.float64(5.545177444479562)}
H (FoR rebuilt):
 [[12.         12.31776617]
 [12.31776617  3.84362411]]
H (Edge-Pushing):
 [[12.         12.31776617]
 [12.31776617  3.84362411]]
diff:
 [[0. 0.]
 [0. 0.]]

==== Test norm_cdf ====
grads: {'x': np.float64(0.24197072451914337)}
H (FoR rebuilt):
 [[-0.24197072]]
H (Edge-Pushing):
 [[-0.24197072]]
diff:
 [[0.]]

==== Test exp+log (mixed) ====
grads: {'x': np.float64(7.88905609893065)}
H (FoR rebuilt):
 [[7.1390561]]
H (Edge-Pushing):
 [[7.1390561

In [2]:
import numpy as np
import pandas as pd

# ==== AAD imports ====
from aad.core.seeds import grads
from aad.core.engine import hvp_for, edge_push_hessian
from aad.ops import exp, log, sqrt, norm_cdf   # <-- use AD-aware ops

# ======================================================
# Pretty printer for Hessians
# ======================================================
def pretty_compare(H_for, H_edge, names, title="Hessian Comparison"):
    diff = H_edge - H_for
    max_diff = float(np.max(np.abs(diff))) if diff.size else 0.0

    print(f"\n==== {title} ====")

    print("\nFoR Hessian:")
    print(pd.DataFrame(H_for, index=names, columns=names).round(6))

    print("\nEdge-Pushing Hessian:")
    print(pd.DataFrame(H_edge, index=names, columns=names).round(6))

    print("\nDifference (EP - FoR):")
    print(pd.DataFrame(diff, index=names, columns=names).round(6))

    print(f"\nMax abs diff = {max_diff:.2e}")
    return diff

# ======================================================
# Check function: compute FoR Hessian and Edge-Pushing Hessian
# ======================================================
def check_f(f, inputs, name="Function"):
    # gradients
    g = grads(f, inputs)
    print(f"\n==== Test {name} ====")
    print("grads:", {k: float(v) for k, v in g.items()})

    ks = list(inputs.keys())
    n = len(ks)

    # FoR: rebuild Hessian via n HVPs (v must be dict!)
    H_for = np.zeros((n, n), dtype=float)
    for j in range(n):
        v = {k: 0.0 for k in ks}
        v[ks[j]] = 1.0
        Hv = hvp_for(f, inputs, v)  # returns array of length n in ks order
        for i in range(n):
            H_for[i, j] = Hv[i]

    # Edge-Pushing (dense)
    H_edge = edge_push_hessian(f, inputs, sparse=False)

    # Pretty print
    diff = pretty_compare(H_for, H_edge, ks, title=name)
    return H_for, H_edge, diff

# ======================================================
# Test 1: Simple function f(x,z,p) = exp(x*z) + log(z+1) + x^p
# ======================================================
f6 = lambda d: exp(d["x"] * d["z"]) + log(d["z"] + 1.0) + (d["x"] ** d["p"])

# ======================================================
# Test 2: BSM Call Option
# ======================================================
def bsm_call(d):
    S = d["S"]; K = d["K"]; T = d["T"]; r = d["r"]; sigma = d["sigma"]
    d1 = (log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return S * norm_cdf(d1) - K * exp(-r * T) * norm_cdf(d2)

# ======================================================
# Main run
# ======================================================
def main():
    # ---- Simple test ----
    check_f(f6, {"x": 1.5, "z": 2.0, "p": 3.0}, "exp(x*z) + log(z+1) + x^p")

    # ---- BSM test ----
    params = {"S": 100.0, "K": 100.0, "T": 1.0, "r": 0.05, "sigma": 0.2}
    check_f(bsm_call, params, "BSM Call Option")

if __name__ == "__main__":
    main()



==== Test exp(x*z) + log(z+1) + x^p ====
grads: {'x': 46.921073846375336, 'z': 30.461638718114834, 'p': 1.3684447398650548}

==== exp(x*z) + log(z+1) + x^p ====

FoR Hessian:
           x          z         p
x  89.342148  80.342148  4.986889
z  80.342148  45.081347  0.000000
p   4.986889   0.000000  0.554857

Edge-Pushing Hessian:
           x          z         p
x  89.342148  80.342148  4.986889
z  80.342148  45.081347  0.000000
p   4.986889   0.000000  0.554857

Difference (EP - FoR):
     x    z    p
x  0.0  0.0  0.0
z  0.0  0.0  0.0
p  0.0  0.0 -0.0

Max abs diff = 1.11e-16

==== Test BSM Call Option ====
grads: {'S': 0.6368306511756191, 'K': -0.5323248154537634, 'T': 6.414027546438197, 'r': 53.23248154537634, 'sigma': 37.52403469169379}

==== BSM Call Option ====

FoR Hessian:
              S         K          T           r      sigma
S      0.018762 -0.018762   0.065667    1.876202  -0.281430
K     -0.018762  0.018762  -0.001527   -1.343877   0.656671
T      0.065667 -0.00152

In [3]:
import numpy as np
from aad.core.seeds import grads
from aad.core.engine import hvp_for, edge_push_hessian
from aad.core.engine import edge_push_hessian_debug, diff_hot_pairs

def localize_mismatch(f, inputs, names=None, top_k=5, tol=1e-8):
    # 1) FoR rebuild Hessian
    ks = list(inputs.keys()) if names is None else list(names)
    n = len(ks)
    H_for = np.zeros((n, n), dtype=float)
    for e, k in enumerate(ks):
        v = {kk: 0.0 for kk in ks}
        v[k] = 1.0
        H_for[:, e] = hvp_for(f, inputs, v)

    # 2) Edge-Pushing Hessian
    H_ep = edge_push_hessian(f, inputs, sparse=False)

    # 3) Find largest-diff pairs
    hot = diff_hot_pairs(H_for, H_ep, ks, top_k=top_k, tol=tol)
    if not hot:
        print("All entries match within tolerance.")
        return

    print("Top mismatched pairs (|diff| desc):")
    for h in hot:
        i, j = h["i"], h["j"]
        print(f"- ({ks[i]},{ks[j]})  diff={h['diff']:.6g}")

    # 4) Focused debug run: only log these pairs
    focus = {(min(h["i"], h["j"]), max(h["i"], h["j"])) for h in hot}
    H_dbg, logs = edge_push_hessian_debug(f, inputs, focus_pairs=focus)

    # 5) Group logs by pair, sort by |contrib|
    from collections import defaultdict
    bucket = defaultdict(list)
    for rec in logs:
        bucket[rec["pair"]].append(rec)

    for pair, recs in bucket.items():
        i, j = pair
        print(f"\n=== Contributions to H[{ks[i]}, {ks[j]}] (EP) ===")
        recs.sort(key=lambda r: abs(r["contrib"]), reverse=True)
        s = 0.0
        for r in recs[:30]: 
            pstr = ",".join([p for p in r["parents"] if p is not None])
            print(f"  node#{r['node_idx']:d}  {r['op_tag']:10s}  {r['kind']:5s}  "
                  f"loc2={r['local_second']:+.6g}  -> contrib={r['contrib']:+.6g}  parents=[{pstr}]")
            s += r["contrib"]
        print(f"  -- sum contrib (logged) = {s:+.6g} ;   EP total = {H_dbg[i,j]:+.6g} ;   FoR = {H_for[i,j]:+.6g}")



In [4]:
def bsm_call(d):
    S = d["S"]; K = d["K"]; T = d["T"]; r = d["r"]; sigma = d["sigma"]
    d1 = (log(S/K) + (r + 0.5 * sigma * sigma) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return S * norm_cdf(d1) - K * exp(-r * T) * norm_cdf(d2)

params = {"S":100.0, "K":100.0, "T":1.0, "r":0.05, "sigma":0.2}
localize_mismatch(bsm_call, params, top_k=5, tol=1e-10)


Top mismatched pairs (|diff| desc):
- (r,r)  diff=187.62
- (sigma,sigma)  diff=15.4787
- (T,T)  diff=0.501884
- (S,S)  diff=0.018762
- (K,K)  diff=0.018762


In [5]:
# ===================== Benchmark helpers (FoR vs Edge-Pushing) =====================
import time
import numpy as np

from aad.core.engine import hvp_for, edge_push_hessian
from aad.core.seeds import grads
from aad.ops import exp, log, sqrt, norm_cdf  # ops we need

def _for_hessian_rebuild(f, inputs):
    """Rebuild the full Hessian via FoR: run n HVPs (one per basis vector)."""
    keys = list(inputs.keys())
    n = len(keys)
    cols = []
    for j in range(n):
        v = {k: 0.0 for k in keys}
        v[keys[j]] = 1.0
        cols.append(hvp_for(f, inputs, v))
    return np.column_stack(cols)

def _time_once(fn, *args, **kwargs):
    """Time a single call to fn(*args, **kwargs) and return (elapsed, result)."""
    t0 = time.perf_counter()
    res = fn(*args, **kwargs)
    t1 = time.perf_counter()
    return (t1 - t0), res

def bench_pair(name, f, inputs, *, warmup=2, runs=10, print_grads=False):
    """
    Benchmark FoR-rebuild vs Edge-Pushing on a given function f and inputs.
    - warmup: number of warmup iterations (not timed)
    - runs  : number of timed iterations (averaged)
    """
    keys = list(inputs.keys())
    n = len(keys)

    # Optional: show first-order grads once to confirm setup
    if print_grads:
        g = grads(f, inputs)
        print(f"[{name}] grads:", g)

    # Warmup
    for _ in range(warmup):
        _ = _for_hessian_rebuild(f, inputs)
        _ = edge_push_hessian(f, inputs, sparse=False)

    # Timed runs: FoR
    for_times = []
    for _ in range(runs):
        dt, _ = _time_once(_for_hessian_rebuild, f, inputs)
        for_times.append(dt)

    # Timed runs: EP
    ep_times = []
    for _ in range(runs):
        dt, _ = _time_once(edge_push_hessian, f, inputs, sparse=False)
        ep_times.append(dt)

    for_avg = float(np.mean(for_times))
    ep_avg  = float(np.mean(ep_times))
    speedup = for_avg / ep_avg if ep_avg > 0 else float("inf")

    print(f"\n=== {name} ===")
    print(f"inputs: {keys} (n={n})")
    print(f"FoR (rebuild via {n} HVPs): {for_avg*1e3:.3f} ms avg over {runs} runs")
    print(f"Edge-Pushing (1 pass)     : {ep_avg*1e3:.3f} ms avg over {runs} runs")
    print(f"Speedup (FoR / EP)        : {speedup:.2f}x")

# ===================== Test set: basic primitives & simple composites =====================

def run_basic_benchmarks():
    # --- Basic primitives (scalar inputs) ---
    bench_pair("mul: f(x,z)=x*z", lambda d: d["x"] * d["z"], {"x": 2.0, "z": 3.0})
    bench_pair("div: f(x,z)=x/z", lambda d: d["x"] / d["z"], {"x": 2.0, "z": 3.0})
    bench_pair("pow: f(x,p)=x**p", lambda d: d["x"] ** d["p"], {"x": 1.5, "p": 2.5})
    bench_pair("norm_cdf: f(x)=N(x)", lambda d: norm_cdf(d["x"]), {"x": 0.5})
    bench_pair("exp+log: f(x)=exp(x)+log(x)", lambda d: exp(d["x"]) + log(d["x"]), {"x": 2.0})

    # --- Simple composite function (the one you used earlier) ---
    # f(x,z,p) = exp(x*z) + log(z+1) + x**p
    def f_combo(d):
        return exp(d["x"] * d["z"]) + log(d["z"] + 1.0) + (d["x"] ** d["p"])

    bench_pair("composite: exp(x*z)+log(z+1)+x**p",
               f_combo,
               {"x": 1.5, "z": 1.5, "p": 2.5},
               runs=10)

# If you want to run immediately:
if __name__ == "__main__":
    run_basic_benchmarks()



=== mul: f(x,z)=x*z ===
inputs: ['x', 'z'] (n=2)
FoR (rebuild via 2 HVPs): 1.724 ms avg over 10 runs
Edge-Pushing (1 pass)     : 0.070 ms avg over 10 runs
Speedup (FoR / EP)        : 24.78x

=== div: f(x,z)=x/z ===
inputs: ['x', 'z'] (n=2)
FoR (rebuild via 2 HVPs): 1.836 ms avg over 10 runs
Edge-Pushing (1 pass)     : 0.072 ms avg over 10 runs
Speedup (FoR / EP)        : 25.67x

=== pow: f(x,p)=x**p ===
inputs: ['x', 'p'] (n=2)
FoR (rebuild via 2 HVPs): 2.058 ms avg over 10 runs
Edge-Pushing (1 pass)     : 0.080 ms avg over 10 runs
Speedup (FoR / EP)        : 25.76x

=== norm_cdf: f(x)=N(x) ===
inputs: ['x'] (n=1)
FoR (rebuild via 1 HVPs): 1.092 ms avg over 10 runs
Edge-Pushing (1 pass)     : 0.075 ms avg over 10 runs
Speedup (FoR / EP)        : 14.47x

=== exp+log: f(x)=exp(x)+log(x) ===
inputs: ['x'] (n=1)
FoR (rebuild via 1 HVPs): 1.182 ms avg over 10 runs
Edge-Pushing (1 pass)     : 0.089 ms avg over 10 runs
Speedup (FoR / EP)        : 13.24x

=== composite: exp(x*z)+log(z+1)+x**p

In [6]:
import time
import numpy as np
from aad.core.engine import hvp_for, edge_push_hessian
from aad.ops import exp, log, sqrt, norm_cdf

# ======= Helper function (timing only) =======
def benchmark_timing(f, inputs, name, repeat=50):
    ks = list(inputs.keys())
    n = len(ks)

    # FoR rebuild timing
    t0 = time.time()
    for _ in range(repeat):
        H_for = np.zeros((n, n))
        for j in range(n):
            v = {k: 0.0 for k in ks}
            v[ks[j]] = 1.0
            Hv = hvp_for(f, inputs, v)
            for i in range(n):
                H_for[i, j] = Hv[i]
    t1 = time.time()
    t_for = (t1 - t0) / repeat

    # Edge-Pushing timing
    t0 = time.time()
    for _ in range(repeat):
        H_ep = edge_push_hessian(f, inputs, sparse=False)
    t1 = time.time()
    t_ep = (t1 - t0) / repeat

    print(f"\n==== Timing {name} ====")
    print(f"FoR={t_for:.6f}s, EP={t_ep:.6f}s")
    if t_ep > 0:
        print(f"Speedup = {t_for/t_ep:.2f}x (EP faster)")
    else:
        print("Speedup = inf (EP ~0s)")

# ======= Test functions =======

def f_mul(d): return d["x"] * d["z"]
def f_div(d): return d["x"] / d["z"]
def f_pow(d): return d["x"] ** d["p"]
def f_cdf(d): return norm_cdf(d["x"])
def f_mix(d): return exp(d["x"]) + log(d["x"] + 1.0)
def f_combo(d): return exp(d["x"] * d["z"]) + log(d["z"] + 1.0) + d["x"] ** d["p"]

# Black–Scholes call
def bsm_call(d):
    S, K, T, r, sigma = d["S"], d["K"], d["T"], d["r"], d["sigma"]
    d1 = (log(S/K) + (r + 0.5*sigma*sigma)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return S * norm_cdf(d1) - K * exp(-r*T) * norm_cdf(d2)

# ======= Run benchmarks =======
def main():
    benchmark_timing(f_mul, {"x": 2.0, "z": 3.0}, "mul")
    benchmark_timing(f_div, {"x": 1.0, "z": 3.0}, "div")
    benchmark_timing(f_pow, {"x": 3.0, "p": 2.5}, "pow")
    benchmark_timing(f_cdf, {"x": 0.5}, "norm_cdf")
    benchmark_timing(f_mix, {"x": 2.0}, "exp+log (mixed)")
    benchmark_timing(f_combo, {"x": 2.0, "z": 1.5, "p": 1.2}, "combo exp+log+pow")
    benchmark_timing(bsm_call, {"S":100.0,"K":100.0,"T":1.0,"r":0.05,"sigma":0.2}, "BSM Call Option")

if __name__ == "__main__":
    main()

  


  



==== Timing mul ====
FoR=0.004158s, EP=0.000156s
Speedup = 26.59x (EP faster)

==== Timing div ====
FoR=0.004969s, EP=0.000173s
Speedup = 28.78x (EP faster)

==== Timing pow ====
FoR=0.005835s, EP=0.000205s
Speedup = 28.50x (EP faster)

==== Timing norm_cdf ====
FoR=0.003284s, EP=0.000216s
Speedup = 15.18x (EP faster)

==== Timing exp+log (mixed) ====
FoR=0.003611s, EP=0.000463s
Speedup = 7.80x (EP faster)

==== Timing combo exp+log+pow ====
FoR=0.015324s, EP=0.000840s
Speedup = 18.24x (EP faster)

==== Timing BSM Call Option ====
FoR=0.063191s, EP=0.002996s
Speedup = 21.09x (EP faster)
